### PERPLEXITY OVERVIEW

https://www.perplexity.ai/search/interretsing-short-form-video-GBzHsz2dSASuXyyYollsEw?login-new=false&login-source=signupButton

### PACKAGES

In [ ]:
%pip install genai
%pip install moviepy
%pip install imageio-ffmpeg
%pip install yt-dlp

### AI EVALUATION

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = os.environ["GOOGLE_API_KEY"]

In [ ]:
from google import genai
from google.genai.types import Part
from pydantic import BaseModel
from google.genai import types
YOUTUBE_URL = "https://www.youtube.com/watch?v=wgaEAzBMLR4"

class Highlights(BaseModel):
    """Schema for Highlights"""
    start_time:str
    end_time: str
    subtitles: str

client = genai.Client()
model_id = "gemini-2.5-flash"

response = client.models.generate_content(
    model=model_id,
    contents=[
        Part.from_uri(
            file_uri=YOUTUBE_URL,
            mime_type="video/mp4",
        ),
        "Extract timestamps (format HH:MM:SS) and subtitles of highligths",
    ],
    config=types.GenerateContentConfig(
        response_schema=list[Highlights],
        response_mime_type="application/json"
      ),
)

In [ ]:
print(response.text)

### YOUTUBE DOWNLOAD

In [ ]:
import yt_dlp

# YouTube video URL
video_url = YOUTUBE_URL

ydl_opts = {} # You can add options here, e.g., for format, quality, etc.
try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([video_url])
    print("Download complete.")
except Exception as e:
    print(f"Error downloading video: {e}")

### EXTRACT AND MERGE CLIPS

In [ ]:
from moviepy.editor import VideoFileClip, concatenate_videoclips
import json
import os
import logging
from pathlib import Path
from typing import List, Dict, Union, Tuple
from concurrent.futures import ThreadPoolExecutor
import tempfile

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class VideoClipExtractor:
    def __init__(self, video_path: str, buffer_seconds: int = 3, temp_dir: str = None):
        self.video_path = Path(video_path)
        self.buffer_seconds = buffer_seconds
        self.temp_dir = temp_dir or tempfile.gettempdir()
        self.video = None
        self.video_duration = None

    def __enter__(self):
        self.video = VideoFileClip(str(self.video_path))
        self.video_duration = self.video.duration
        logger.info(f"Loaded video: {self.video_path.name} (Duration: {self.video_duration:.2f}s)")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.video:
            self.video.close()

    @staticmethod
    def hms_to_seconds(hms_string: str) -> float:
        """Converts HH:MM:SS, MM:SS, or SS format to seconds with decimal support."""
        parts = hms_string.strip().split(':')

        if len(parts) == 1:  # SS or SS.milliseconds
            return float(parts[0])
        elif len(parts) == 2:  # MM:SS
            minutes, seconds = map(float, parts)
            return minutes * 60 + seconds
        elif len(parts) == 3:  # HH:MM:SS
            hours, minutes, seconds = map(float, parts)
            return hours * 3600 + minutes * 60 + seconds
        else:
            raise ValueError(f"Invalid timestamp format: {hms_string}")

    def validate_and_adjust_times(self, start_time: float, end_time: float) -> Tuple[float, float]:
        """Validates and adjusts clip times to be within video bounds."""
        # Add buffer and clamp to video bounds
        adjusted_start = max(0, start_time - self.buffer_seconds)
        adjusted_end = min(self.video_duration, end_time + self.buffer_seconds)

        # Ensure minimum clip duration of 1 second
        if adjusted_end - adjusted_start < 1:
            adjusted_end = min(self.video_duration, adjusted_start + 1)

        return adjusted_start, adjusted_end

    def extract_clips_in_memory(self, clip_data: List[Dict]) -> List[VideoFileClip]:
        """Extracts clips and keeps them in memory for faster merging."""
        clips = []

        for i, data in enumerate(clip_data):
            try:
                start_time = self.hms_to_seconds(data['start_time'])
                end_time = self.hms_to_seconds(data['end_time'])

                # Validate times
                if start_time >= end_time:
                    logger.warning(f"Clip {i+1}: Invalid time range ({start_time}s >= {end_time}s), skipping")
                    continue

                adjusted_start, adjusted_end = self.validate_and_adjust_times(start_time, end_time)

                logger.info(f"Extracting clip {i+1}: {adjusted_start:.2f}s to {adjusted_end:.2f}s")

                # Extract subclip
                subclip = self.video.subclip(adjusted_start, adjusted_end)
                clips.append(subclip)

            except (KeyError, ValueError) as e:
                logger.error(f"Clip {i+1}: Invalid data format - {e}")
                continue
            except Exception as e:
                logger.error(f"Clip {i+1}: Extraction failed - {e}")
                continue

        return clips

    def merge_clips_optimized(self, clips: List[VideoFileClip], output_path: str,
                            quality: str = "medium") -> bool:
        """Merges clips with optimized settings and audio bug workarounds."""
        if not clips:
            logger.error("No clips to merge")
            return False

        try:
            # Quality presets
            quality_settings = {
                "low": {"bitrate": "500k", "preset": "ultrafast"},
                "medium": {"bitrate": "1500k", "preset": "fast"},
                "high": {"bitrate": "3000k", "preset": "medium"}
            }

            settings = quality_settings.get(quality, quality_settings["medium"])

            logger.info(f"Merging {len(clips)} clips with {quality} quality...")

            # Method 1: Try with audio handling fix
            try:
                final_video = concatenate_videoclips(clips, method="compose")

                # Fix for FFMPEG_AudioWriter bug - use temp_audiofile_path instead
                temp_audio_path = os.path.join(self.temp_dir, f"temp_audio_{os.getpid()}.m4a")

                final_video.write_videofile(
                    output_path,
                    codec="libx264",
                    audio_codec="aac",
                    bitrate=settings["bitrate"],
                    preset=settings["preset"],
                    temp_audiofile=temp_audio_path,
                    remove_temp=True,
                    verbose=False,
                    logger=None
                )

                final_video.close()
                logger.info(f"Merged video saved: {output_path}")
                return True

            except Exception as audio_error:
                logger.warning(f"Audio merge failed, trying without audio: {audio_error}")

                # Method 2: Fallback - merge without audio
                try:
                    # Remove audio from clips
                    video_only_clips = [clip.without_audio() for clip in clips]
                    final_video = concatenate_videoclips(video_only_clips, method="compose")

                    final_video.write_videofile(
                        output_path,
                        codec="libx264",
                        bitrate=settings["bitrate"],
                        preset=settings["preset"],
                        verbose=False,
                        logger=None
                    )

                    final_video.close()
                    for clip in video_only_clips:
                        clip.close()

                    logger.info(f"Merged video (no audio) saved: {output_path}")
                    return True

                except Exception as video_error:
                    logger.error(f"Video-only merge also failed: {video_error}")

                    # Method 3: Last resort - sequential file-based merge
                    return self._merge_via_files(clips, output_path, settings)

        except Exception as e:
            logger.error(f"Merging failed: {e}")
            return False
        finally:
            # Cleanup clips
            for clip in clips:
                try:
                    clip.close()
                except:
                    pass

    def _merge_via_files(self, clips: List[VideoFileClip], output_path: str, settings: dict) -> bool:
        """Fallback method: save clips to temp files then merge."""
        temp_files = []

        try:
            logger.info("Using file-based merge as fallback...")

            # Save clips to temporary files
            for i, clip in enumerate(clips):
                temp_file = os.path.join(self.temp_dir, f"temp_clip_{i:03d}.mp4")
                clip.write_videofile(
                    temp_file,
                    codec="libx264",
                    verbose=False,
                    logger=None
                )
                temp_files.append(temp_file)
                clip.close()

            # Load temp files and merge
            temp_clips = [VideoFileClip(f) for f in temp_files]
            final_video = concatenate_videoclips(temp_clips, method="compose")

            final_video.write_videofile(
                output_path,
                codec="libx264",
                bitrate=settings["bitrate"],
                preset=settings["preset"],
                verbose=False,
                logger=None
            )

            final_video.close()
            for clip in temp_clips:
                clip.close()

            logger.info(f"File-based merge completed: {output_path}")
            return True

        except Exception as e:
            logger.error(f"File-based merge failed: {e}")
            return False
        finally:
            # Clean up temp files
            for temp_file in temp_files:
                try:
                    if os.path.exists(temp_file):
                        os.remove(temp_file)
                except:
                    pass

def process_video_highlights(video_path: str, json_data: Union[str, List[Dict]],
                           output_path: str = None, quality: str = "medium",
                           save_individual_clips: bool = False) -> bool:
    """
    Main function to extract and merge video highlights.

    Args:
        video_path: Path to input video
        json_data: JSON string or list with clip data
        output_path: Output path for merged video
        quality: Video quality ("low", "medium", "high")
        save_individual_clips: Whether to save individual clips
    """

    # Parse JSON if string
    if isinstance(json_data, str):
        try:
            clip_data = json.loads(json_data)
        except json.JSONDecodeError as e:
            logger.error(f"Invalid JSON data: {e}")
            return False
    else:
        clip_data = json_data

    # Set default output path
    if not output_path:
        video_name = Path(video_path).stem
        output_path = f"{video_name}_highlights.mp4"

    # Process with context manager for automatic cleanup
    with VideoClipExtractor(video_path) as extractor:
        # Extract clips in memory
        clips = extractor.extract_clips_in_memory(clip_data)

        if not clips:
            logger.error("No valid clips extracted")
            return False

        # Save individual clips if requested
        if save_individual_clips:
            logger.info("Saving individual clips...")
            for i, clip in enumerate(clips):
                individual_path = f"clip_{i+1:03d}.mp4"
                try:
                    clip.write_videofile(individual_path, codec="libx264", verbose=False, logger=None)
                    logger.info(f"Saved: {individual_path}")
                except Exception as e:
                    logger.error(f"Failed to save individual clip {i+1}: {e}")

        # Merge clips
        success = extractor.merge_clips_optimized(clips, output_path, quality)

        return success

# Parallel processing version for very large numbers of clips
def extract_clips_parallel(video_path: str, clip_data: List[Dict], max_workers: int = 4) -> List[str]:
    """Extract clips in parallel and save to temporary files."""

    def extract_single_clip(args):
        i, data, video_path = args
        try:
            start_time = VideoClipExtractor.hms_to_seconds(data['start_time']) - 3
            end_time = VideoClipExtractor.hms_to_seconds(data['end_time']) + 3

            with VideoFileClip(video_path) as video:
                start_time = max(0, start_time)
                end_time = min(video.duration, end_time)

                temp_path = f"temp_clip_{i:03d}.mp4"
                subclip = video.subclip(start_time, end_time)
                subclip.write_videofile(temp_path, codec="libx264", verbose=False, logger=None)
                subclip.close()

                return temp_path

        except Exception as e:
            logger.error(f"Parallel extraction failed for clip {i}: {e}")
            return None

    # Prepare arguments for parallel processing
    args_list = [(i, data, video_path) for i, data in enumerate(clip_data)]

    # Extract in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        temp_files = list(executor.map(extract_single_clip, args_list))

    # Filter out failed extractions
    return [f for f in temp_files if f is not None]

# Example usage and main execution
if __name__ == "__main__":
    # Video file path
    video_path = 'Edgar Berlanga vs. Hamzah Sheeraz ｜ FIGHT HIGHLIGHTS [wgaEAzBMLR4].webm'

    # Example JSON data - replace with your actual data
    example_data = json.loads(response.text)

    # Method 1: Standard processing (recommended for most cases)
    print("=== Standard Processing ===")
    success = process_video_highlights(
        video_path=video_path,
        json_data=example_data,
        output_path="fight_highlights.mp4",
        quality="medium",
        save_individual_clips=False
    )

    if success:
        print("✅ Video processing completed successfully!")
    else:
        print("❌ Video processing failed!")

    # Method 2: Parallel processing (for large numbers of clips)
    # Uncomment below for parallel processing
    """
    print("\n=== Parallel Processing ===")
    temp_files = extract_clips_parallel(video_path, example_data, max_workers=4)

    if temp_files:
        # Load and merge temp files
        temp_clips = [VideoFileClip(f) for f in temp_files]
        final_video = concatenate_videoclips(temp_clips)
        final_video.write_videofile("parallel_highlights.mp4", codec="libx264")

        # Cleanup
        final_video.close()
        for clip in temp_clips:
            clip.close()
        for temp_file in temp_files:
            os.remove(temp_file)

        print("✅ Parallel processing completed!")
    """



TO GENERATE SUBTITLES LIKE THIS WITH OPENSOURCE OPTIONS
https://chatgpt.com/share/68b50055-3d3c-800b-82ee-ab21e3e7d0b5